# 1.2. XỬ LÝ VÀ TRỰC QUAN HÓA DỮ LIỆU
**Lab 05 – Phân tích khám phá dữ liệu (EDA)** · Trường Đại học Sài Gòn

Notebook gồm:
- **1.2.1** Ôn tập lý thuyết (kèm code mẫu vẽ và xuất biểu đồ)
- **1.2.2** Bài tập thực hành 1 – trực quan hóa Red Wine Quality
- **1.2.3** Bài tập thực hành 2 – trực quan hóa Pima Indians Diabetes và EDA Online Retail

📦 **Dữ liệu được nạp trực tiếp từ Kaggle** bằng thư viện `kagglehub` (hàm `kaggle_read`), không cần tải file thủ công.
Lần chạy đầu cần có mạng; nếu Kaggle yêu cầu đăng nhập, thông báo lỗi sẽ hướng dẫn cách đăng nhập (`kagglehub.login()`).
Ảnh biểu đồ xuất ra được lưu trong thư mục `output/` (tự tạo).

In [ ]:
import os, sys, subprocess
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


# Dataset trên Kaggle (được nạp trực tiếp bằng kagglehub, không cần tải file thủ công)
KG_WINE = "uciml/red-wine-quality-cortez-et-al-2009"      # https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009
KG_DIAB = ["uciml/pima-indians-diabetes-database",        # https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
           "akshaydattatraykhare/diabetes-dataset",       # bản sao dự phòng trên Kaggle (cùng file diabetes.csv)
           "mathchi/diabetes-data-set"]                   # bản sao dự phòng trên Kaggle
# Nếu Kaggle không truy cập được (403, cần đăng nhập...), hàm kaggle_read sẽ dùng bản sao công khai trên GitHub:
URL_WINE_BACKUP = "https://raw.githubusercontent.com/plotly/datasets/master/winequality-red.csv"
URL_DIAB_BACKUP = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
KG_RETAIL = ["vijayuv/onlineretail", "carrie1/ecommerce-data"]   # Online Retail – thử lần lượt các dataset


def kaggle_read(slugs, filename, fallback_url=None, **kwargs):
    """Nạp dataset trực tiếp từ Kaggle thành DataFrame (kagglehub tự tải và lưu cache, không cần tải tay).
    slugs: 'chu-so-huu/ten-dataset' hoặc danh sách slug để thử lần lượt; filename: tên file CSV cần đọc.
    fallback_url: (tuỳ chọn) link CSV công khai, chỉ dùng khi Kaggle không truy cập được."""
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    if isinstance(slugs, str):
        slugs = [slugs]
    last_err = None
    for slug in slugs:
        try:
            folder = kagglehub.dataset_download(slug)
            files = [os.path.join(r, f) for r, _, fs in os.walk(folder) for f in fs if f.lower().endswith(".csv")]
            if not files:
                continue
            same = [f for f in files if os.path.basename(f).lower() == filename.lower()]
            path = same[0] if same else max(files, key=os.path.getsize)
            print(f"✅ Đã nạp dữ liệu từ Kaggle: {slug}  →  {os.path.basename(path)}")
            try:
                return pd.read_csv(path, encoding="utf-8-sig", **kwargs)
            except UnicodeDecodeError:
                return pd.read_csv(path, encoding="latin-1", **kwargs)
        except Exception as e:
            last_err = e
            print(f"   ⚠️ Không nạp được từ '{slug}': {repr(e)[:200]}")
    if fallback_url:
        print(f"↪️  Kaggle không truy cập được → dùng bản sao công khai: {fallback_url}")
        return pd.read_csv(fallback_url, encoding="utf-8-sig", **kwargs)
    raise RuntimeError(
        "Không nạp được dữ liệu từ Kaggle.\n"
        "  • Kiểm tra kết nối mạng.\n"
        "  • Nếu Kaggle yêu cầu đăng nhập: chạy `import kagglehub; kagglehub.login()` (cần API token: kaggle.com → Settings → API),\n"
        "    hoặc đặt biến môi trường KAGGLE_USERNAME và KAGGLE_KEY rồi chạy lại.\n"
        f"  Lỗi cuối: {last_err!r}"
    )


def describe_full(df):
    """Bảng thống kê mô tả đầy đủ cho các cột số."""
    num = df.select_dtypes(include="number")
    q1, q3 = num.quantile(0.25), num.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    res = pd.DataFrame({
        "count": num.count(),
        "missing": num.isna().sum(),
        "mean": num.mean(),
        "median": num.median(),
        "mode": num.mode().iloc[0],
        "var": num.var(),
        "std": num.std(),
        "min": num.min(),
        "max": num.max(),
        "range": num.max() - num.min(),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "skew": num.skew(),
        "kurtosis": num.kurt(),
        "n_outliers(IQR)": ((num < low) | (num > high)).sum(),
    })
    return res

os.makedirs("output", exist_ok=True)          # nơi lưu ảnh biểu đồ xuất ra
sns.set_theme()

---
# 1.2.1. Ôn tập lý thuyết

**1. Vai trò của trực quan hóa trong phân tích dữ liệu? Vì sao quan trọng trong EDA?**
Con người nhận ra hình dạng, xu hướng, cụm, ngoại lai nhanh hơn nhiều so với đọc bảng số. Trong EDA, biểu đồ giúp phát hiện lỗi dữ liệu, kiểm tra giả định, tìm quan hệ giữa các biến và định hướng bước phân tích/mô hình tiếp theo. Ví dụ kinh điển: các tập dữ liệu có cùng mean/std/tương quan (Anscombe's quartet) nhưng hình dạng hoàn toàn khác nhau.

**2. Các loại biểu đồ phổ biến dùng khi nào?**

| Biểu đồ | Dùng cho |
|---|---|
| Histogram | Phân bố của **một biến số** liên tục |
| Boxplot / Violin | Tóm tắt phân bố, phát hiện ngoại lai, so sánh giữa các nhóm |
| Bar chart | So sánh giá trị/số lượng giữa các **nhóm (biến phân loại)** |
| Scatter plot | Quan hệ giữa **hai biến số** |
| Line chart | Xu hướng theo **thời gian** |
| Heatmap | Ma trận tương quan, bảng đếm chéo |

**3. Chọn biểu đồ theo đặc điểm dữ liệu?**
- *Phân loại*: bar/count plot (pie chỉ khi ít nhóm và cần thể hiện tỉ lệ trên tổng).
- *Số (1 biến)*: histogram, KDE, boxplot. *Số × số*: scatter, heatmap tương quan. *Số × phân loại*: boxplot/violin/bar theo nhóm.
- *Thời gian*: line chart (có thể gộp theo ngày/tháng).

**4. Matplotlib, Seaborn, Plotly khác nhau thế nào?**
- *Matplotlib*: nền tảng, kiểm soát chi tiết nhất nhưng viết dài.
- *Seaborn*: xây trên Matplotlib, giao diện đẹp mặc định, làm việc trực tiếp với DataFrame, có sẵn biểu đồ thống kê (hist, box, violin, heatmap, pairplot…).
- *Plotly*: biểu đồ **tương tác** (zoom, hover), xuất HTML, hợp với dashboard/web.

**5. Nguyên tắc thiết kế biểu đồ dễ hiểu?**
Có tiêu đề, nhãn trục và đơn vị rõ ràng; chọn đúng loại biểu đồ; thang đo trung thực (bar chart nên bắt đầu từ 0); hạn chế màu/hiệu ứng thừa, tránh 3D; sắp xếp thứ tự có ý nghĩa; chú thích (legend) khi cần; chọn màu thân thiện với người mù màu; mỗi biểu đồ nên nói một thông điệp.

**6–7.** Code mẫu vẽ histogram, bar chart bằng Matplotlib và xuất ra PNG / PDF / HTML ở hai cell bên dưới.

In [ ]:
# Code mẫu: histogram và bar chart bằng Matplotlib (dùng dữ liệu giả lập, chưa cần tải dữ liệu)
rng = np.random.default_rng(42)
scores = rng.normal(loc=65, scale=12, size=300)                                  # dữ liệu số
cities = rng.choice(["Hà Nội", "TP.HCM", "Đà Nẵng", "Cần Thơ"], size=300, p=[.3, .4, .2, .1])  # dữ liệu phân loại

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores, bins=15, edgecolor="black")
axes[0].set_title("Histogram – phân bố điểm")
axes[0].set_xlabel("Điểm")
axes[0].set_ylabel("Số lượng")

counts = pd.Series(cities).value_counts()
axes[1].bar(counts.index, counts.values, color="tab:orange")
axes[1].set_title("Bar chart – số lượng theo thành phố")
axes[1].set_xlabel("Thành phố")
axes[1].set_ylabel("Số lượng")

plt.tight_layout()

# Xuất biểu đồ ra file để đưa vào báo cáo
fig.savefig("output/demo_chart.png", dpi=200, bbox_inches="tight")   # PNG (ảnh)
fig.savefig("output/demo_chart.pdf", bbox_inches="tight")            # PDF (vector, in ấn)
plt.show()
print("Đã lưu: output/demo_chart.png và output/demo_chart.pdf")

In [ ]:
# Xuất HTML tương tác bằng Plotly (cần: pip install plotly)
try:
    import plotly.express as px
    fig_px = px.bar(x=counts.index, y=counts.values,
                    labels={"x": "Thành phố", "y": "Số lượng"}, title="Bar chart tương tác")
    fig_px.write_html("output/demo_chart.html")
    print("Đã lưu: output/demo_chart.html (mở bằng trình duyệt)")
except ImportError:
    print("Chưa cài plotly → chạy: pip install plotly")

---
# 1.2.2. Bài tập thực hành 1 – Trực quan hóa dữ liệu Red Wine Quality

📦 **Dữ liệu (Kaggle):** https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009
(notebook tham khảo: https://www.kaggle.com/code/eisgandar/red-wine-quality-eda-classification)
→ nạp trực tiếp bằng `kaggle_read` ở cell dưới.

In [ ]:
wine = kaggle_read(KG_WINE, "winequality-red.csv", fallback_url=URL_WINE_BACKUP, sep=None, engine="python")
print(wine.shape)
display(wine.head())

In [ ]:
# a) Biến mục tiêu: phân bố của quality
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=wine, x="quality")
ax.bar_label(ax.containers[0])
ax.set_title("Phân bố chất lượng rượu (quality)", fontsize=15)
ax.set_xlabel("quality")
ax.set_ylabel("Số lượng")
plt.show()

In [ ]:
# b) Phân tích đơn biến: histogram + KDE cho từng đặc trưng
features = wine.columns.drop("quality")
n_cols = 4
n_rows = int(np.ceil(len(features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(wine[col], kde=True, ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# c) Boxplot từng đặc trưng → phát hiện ngoại lai
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(x=wine[col], ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# d) Quan hệ giữa đặc trưng và quality (biến số × biến phân loại)
key_feats = ["alcohol", "volatile acidity", "sulphates", "citric acid"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, col in zip(axes, key_feats):
    sns.boxplot(data=wine, x="quality", y=col, ax=ax)
    ax.set_title(f"{col} theo quality")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.violinplot(data=wine, x="quality", y="alcohol")
plt.title("Phân bố alcohol theo quality (violin)")
plt.show()

In [ ]:
# e) Heatmap tương quan
plt.figure(figsize=(12, 9))
sns.heatmap(wine.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Ma trận tương quan – Red Wine", fontsize=15)
plt.savefig("output/wine_corr_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# f) Scatter plot một số cặp biến, tô màu theo quality
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.scatterplot(data=wine, x="alcohol", y="density", hue="quality", palette="viridis", ax=axes[0])
sns.scatterplot(data=wine, x="fixed acidity", y="pH", hue="quality", palette="viridis", ax=axes[1])
sns.scatterplot(data=wine, x="free sulfur dioxide", y="total sulfur dioxide", hue="quality", palette="viridis", ax=axes[2])
plt.tight_layout()
plt.show()

# Pairplot cho nhóm đặc trưng quan trọng
sns.pairplot(wine[["alcohol", "volatile acidity", "sulphates", "citric acid", "quality"]], hue="quality", palette="viridis", corner=True)
plt.show()

**Gợi ý nhận xét:** `quality` tập trung ở mức 5–6 (mất cân bằng); `alcohol` cao thường đi cùng `quality` cao, `volatile acidity` cao đi cùng `quality` thấp;
`fixed acidity`–`citric acid`–`density`–`pH` tương quan với nhau (đa cộng tuyến); `residual sugar`, `chlorides`, `sulphates` lệch phải và nhiều ngoại lai.

---
# 1.2.3. Bài tập thực hành 2
## (a) Trực quan hóa dữ liệu bệnh tiểu đường (Pima Indians)

📦 **Dữ liệu (Kaggle):** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
(notebook tham khảo: https://www.kaggle.com/code/vincentlugat/pima-indians-diabetes-eda-prediction-0-906)
→ nạp trực tiếp bằng `kaggle_read` ở cell dưới.

In [ ]:
diab = kaggle_read(KG_DIAB, "diabetes.csv", fallback_url=URL_DIAB_BACKUP)

# Các cột sau không thể bằng 0 về mặt sinh học → 0 thực chất là giá trị thiếu
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
diab[zero_cols] = diab[zero_cols].replace(0, np.nan)
print(diab.shape)
display(diab.isna().sum().to_frame("missing"))

In [ ]:
# a) Phân bố biến mục tiêu Outcome (bar chart + pie chart)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=diab, x="Outcome", ax=axes[0])
axes[0].set_title("Số lượng theo Outcome")
axes[0].bar_label(axes[0].containers[0])

counts = diab["Outcome"].value_counts().sort_index()
axes[1].pie(counts, labels=["Không bệnh (0)", "Có bệnh (1)"], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Tỉ lệ Outcome")
plt.tight_layout()
plt.show()

In [ ]:
# b) Histogram từng đặc trưng, tách theo Outcome
feats = diab.columns.drop("Outcome")
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.ravel(), feats):
    sns.histplot(data=diab, x=col, hue="Outcome", kde=True, ax=ax, stat="density", common_norm=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# c) Boxplot từng đặc trưng theo Outcome
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.ravel(), feats):
    sns.boxplot(data=diab, x="Outcome", y=col, ax=ax)
    ax.set_title(f"{col} theo Outcome")
plt.tight_layout()
plt.show()

In [ ]:
# d) Heatmap tương quan
plt.figure(figsize=(9, 7))
sns.heatmap(diab.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Ma trận tương quan – Pima Indians Diabetes", fontsize=14)
plt.savefig("output/diabetes_corr_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# e) Scatter / pairplot cho các biến quan trọng
plt.figure(figsize=(8, 6))
sns.scatterplot(data=diab, x="Glucose", y="BMI", hue="Outcome", alpha=0.7)
plt.title("Glucose vs BMI theo Outcome")
plt.show()

sns.pairplot(diab[["Glucose", "BMI", "Age", "Insulin", "Outcome"]], hue="Outcome", corner=True)
plt.show()

# Tuổi theo nhóm → tỉ lệ mắc bệnh
diab["AgeGroup"] = pd.cut(diab["Age"], bins=[20, 30, 40, 50, 60, 100], labels=["21-30", "31-40", "41-50", "51-60", "60+"])
plt.figure(figsize=(8, 5))
sns.barplot(data=diab, x="AgeGroup", y="Outcome", errorbar=None)
plt.title("Tỉ lệ mắc bệnh theo nhóm tuổi")
plt.ylabel("Tỉ lệ Outcome = 1")
plt.show()

---
## (b) EDA tập dữ liệu mua sắm tại siêu thị (Online Retail)

📦 **Dữ liệu (Kaggle):** dataset Online Retail dùng trong notebook https://www.kaggle.com/code/rajatkumar30/eda-online-retail
→ nạp trực tiếp bằng `kaggle_read` ở cell dưới (thử lần lượt các dataset trong `KG_RETAIL`).

> File có ký tự đặc biệt nên hàm tự thử `utf-8` rồi chuyển sang `latin-1`.
> Các cột dự kiến: `InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country`.

In [ ]:
retail = kaggle_read(KG_RETAIL, "OnlineRetail.csv")
print(retail.shape)
display(retail.head())
retail.info()

print("\nGiá trị thiếu:")
display(retail.isna().sum().to_frame("missing").assign(percent=lambda d: (d["missing"] / len(retail) * 100).round(2)))
print("Số dòng trùng lặp:", retail.duplicated().sum())

In [ ]:
# Làm sạch dữ liệu
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"])

is_cancel = retail["InvoiceNo"].astype(str).str.startswith("C")       # hóa đơn huỷ bắt đầu bằng chữ C
print("Số hóa đơn huỷ:", is_cancel.sum())

sales = retail[(~is_cancel) & (retail["Quantity"] > 0) & (retail["UnitPrice"] > 0)].drop_duplicates().copy()
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["Month"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
sales["Hour"] = sales["InvoiceDate"].dt.hour
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()

print("Sau làm sạch:", sales.shape)
display(sales[["Quantity", "UnitPrice", "Revenue"]].describe())

In [ ]:
# 1) Top 10 quốc gia theo doanh thu
top_country = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(11, 5))
sns.barplot(x=top_country.values, y=top_country.index)
plt.title("Top 10 quốc gia theo doanh thu", fontsize=15)
plt.xlabel("Doanh thu")
plt.ylabel("")
plt.ticklabel_format(style="plain", axis="x")
plt.show()

# Bỏ nước chiếm đa số để thấy rõ phần còn lại
top_country_wo_top = top_country.iloc[1:]
plt.figure(figsize=(11, 5))
sns.barplot(x=top_country_wo_top.values, y=top_country_wo_top.index)
plt.title(f"Top quốc gia theo doanh thu (loại bỏ {top_country.index[0]})", fontsize=15)
plt.ticklabel_format(style="plain", axis="x")
plt.show()

In [ ]:
# 2) Top 10 sản phẩm bán chạy (theo số lượng) và theo doanh thu
top_qty = sales.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)
top_rev = sales.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(22, 6))
sns.barplot(x=top_qty.values, y=top_qty.index, ax=axes[0])
axes[0].set_title("Top 10 sản phẩm theo số lượng bán")
sns.barplot(x=top_rev.values, y=top_rev.index, ax=axes[1])
axes[1].set_title("Top 10 sản phẩm theo doanh thu")
plt.tight_layout()
plt.show()

In [ ]:
# 3) Xu hướng doanh thu theo tháng
monthly = sales.groupby("Month")["Revenue"].sum()
plt.figure(figsize=(12, 5))
plt.plot(monthly.index, monthly.values, marker="o")
plt.title("Doanh thu theo tháng", fontsize=15)
plt.xlabel("Tháng")
plt.ylabel("Doanh thu")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 4) Thời điểm mua hàng: theo giờ và theo thứ trong tuần (số hóa đơn)
by_hour = sales.groupby("Hour")["InvoiceNo"].nunique()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = sales.groupby("Weekday")["InvoiceNo"].nunique().reindex(order).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=by_hour.index, y=by_hour.values, ax=axes[0], color="tab:blue")
axes[0].set_title("Số hóa đơn theo giờ")
axes[0].set_xlabel("Giờ")
sns.barplot(x=by_day.index, y=by_day.values, ax=axes[1], color="tab:green")
axes[1].set_title("Số hóa đơn theo thứ trong tuần")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# 5) Phân bố giá trị mỗi hóa đơn (lệch phải mạnh → dùng thang log)
invoice_value = sales.groupby("InvoiceNo")["Revenue"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(invoice_value, bins=50, log_scale=True, ax=axes[0])
axes[0].set_title("Phân bố giá trị hóa đơn (log)")
sns.boxplot(x=invoice_value, ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_title("Boxplot giá trị hóa đơn (log)")
plt.tight_layout()
plt.show()

# 6) Khách hàng chi tiêu nhiều nhất
top_cust = sales.dropna(subset=["CustomerID"]).groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_cust.index.astype(int).astype(str), y=top_cust.values)
plt.title("Top 10 khách hàng theo doanh thu")
plt.xlabel("CustomerID")
plt.ticklabel_format(style="plain", axis="y")
plt.show()

**Gợi ý nhận xét:** doanh thu tập trung rất lớn ở một quốc gia (thường là United Kingdom); doanh thu tăng mạnh vào các tháng cuối năm (mua sắm dịp lễ);
phần lớn hóa đơn phát sinh vào khoảng giữa trưa và các ngày trong tuần; giá trị hóa đơn lệch phải mạnh với một số ít hóa đơn rất lớn (khách sỉ).

---
## ✅ Tóm tắt
- Biểu đồ phải **đúng loại** với kiểu dữ liệu (phân loại → bar; số → histogram/box; số × số → scatter/heatmap; thời gian → line).
- Luôn có tiêu đề, nhãn trục; dùng thang log khi dữ liệu lệch mạnh.
- Xuất biểu đồ bằng `fig.savefig(...)` (PNG/PDF) hoặc `write_html` của Plotly để đưa vào báo cáo.